In [23]:
from pathlib import Path
import csv
from datetime import date
import sqlite3

In [24]:
master_db_path = Path("../databases/Master_Database.db").resolve()
export_folder = Path("../lab batches/flat_01")
master_conn = sqlite3.connect(master_db_path)


local_flat_folder = Path("../lab batches/flatfields")
lab_flat_folder = "/home/astrolab/SNe/flatfields"

In [25]:
def get_mosaic_work(master_conn):
   sql = """
       SELECT group_id, object, obs_date, telescope,
              filter, img_start, img_end
       FROM observations
       WHERE mosaic_name_f IS NULL
         AND error_f IS NULL
       ORDER BY group_id
   """

   work_list = master_conn.execute(sql).fetchall()

   return work_list

In [26]:
flat_file_names = {
    ("pt5m", "B"): "Bflatnorm2x2.fits",
    ("pt5m", "V"): "Vflatnorm2x2.fits",
    ("pt5m", "R"): "Rflatnorm2x2.fits",
    ("pt5m", "I"): "Iflatnorm2x2.fits",
    ("east-16", "B"): "master_flat_east-16_26_10_07_B.fits",
    ("east-16", "V"): "master_flat_east-16_26_10_09_V.fits",
    ("east-16", "r"): "master_flat_east-16_26_10_09_r.fits",
    ("west-14", "V"): "master_flat_west-14_26_10_01_V.fits",
    ("west-14", "R"): "master_flat_west-14_26_10_01_R.fits",
    ("draco2", "V"): "master_flat_draco2_26_10_22_V.fits",
    ("draco2", "C"): "master_flat_draco2_26_10_22_C.fits",
}

In [27]:
def find_flat(telescope, band):
    flat_name = flat_file_names.get((telescope, band))

    if flat_name is None:
        return None

    return f"{lab_flat_folder}/{flat_name}"

In [28]:
work_list = get_mosaic_work(master_conn)

ready_list = []
skipped_counts = {}

for row in work_list:
    group_id, object_name, obs_date, telescope, band, img_start, img_end = row

    flat_path = find_flat(telescope, band)

    if flat_path is None:
        key = (telescope, band)
        skipped_counts[key] = skipped_counts.get(key, 0) + 1
    else:
        ready_list.append(row + (flat_path,))

print("Groups needing flat mosaics:", len(work_list))
print("Ready to export:", len(ready_list))
print("Skipped, no flat:", len(work_list) - len(ready_list))

for key, count in sorted(skipped_counts.items()):
    print("  ", key, count)

for row in ready_list[:5]:
    print(row)

Groups needing flat mosaics: 194
Ready to export: 186
Skipped, no flat: 8
   ('draco2', 'B') 3
   ('draco2', 'r') 4
   ('west-14', 'B') 1
(1, 'AT2026AEIZ', '2026-10-13', 'east-16', 'V', '0146', '0146', '/home/astrolab/SNe/flatfields/master_flat_east-16_26_10_09_V.fits')
(2, 'AT2026AESJ', '2026-10-13', 'pt5m', 'B', '0884578', '0884585', '/home/astrolab/SNe/flatfields/Bflatnorm2x2.fits')
(3, 'AT2026AESJ', '2026-10-13', 'pt5m', 'V', '0884586', '0884593', '/home/astrolab/SNe/flatfields/Vflatnorm2x2.fits')
(4, 'AT2026AESJ', '2026-10-13', 'pt5m', 'R', '0884594', '0884601', '/home/astrolab/SNe/flatfields/Rflatnorm2x2.fits')
(5, 'AT2026AESJ', '2026-10-13', 'pt5m', 'I', '0884602', '0884609', '/home/astrolab/SNe/flatfields/Iflatnorm2x2.fits')


In [29]:
def make_mosaic_commands(group_id, telescope, lab_date,
                        img_start, img_end, filename, flat_path):

   return f'''
echo "Processing group {group_id}"

if ( -e "$batch_dir/output_mosaics/{filename}" ) then
   echo "Output already exists; not overwriting group {group_id}"

else if ( -d "$batch_dir/working/g{group_id}" ) then
   echo "Work folder already exists; inspect logs/g{group_id}.log"

else if ( ! -f "{flat_path}" ) then
   echo "Missing flat for group {group_id}: {flat_path}"

else
   mkdir "$batch_dir/working/g{group_id}"
   if ( $status != 0 ) exit 1

   cd "$batch_dir/working/g{group_id}"
   if ( $status != 0 ) exit 1

   python /mnt/64bin/stackX2.py "{telescope}" "{lab_date}" "{img_start}" "{img_end}" 5.0 "{flat_path}" >&! "$batch_dir/logs/g{group_id}.log"
   set stack_status = $status

   grep -q N_ASTARS amosaic.fits >& /dev/null
   set astrom_status = $status

   grep -q -F "Flatfield found: {flat_path}" "$batch_dir/logs/g{group_id}.log" >& /dev/null
   set flat_status = $status

   if ( $stack_status == 0 && $astrom_status == 0 && $flat_status == 0 ) then
       cp amosaic.fits "$batch_dir/output_mosaics/{filename}"
       if ( $status != 0 ) exit 1

       echo "Saved group {group_id}"
   else
       echo "Failed group {group_id}; see logs/g{group_id}.log"
       if ( $flat_status != 0 ) echo "Flat was not applied in group {group_id}"
   endif

   cd "$batch_dir"
   if ( $status != 0 ) exit 1
endif
'''

In [30]:
def write_mosaic_script(ready_list, export_folder):
    export_folder.mkdir(parents=True, exist_ok=True)

    output = [
        "#!/bin/csh -f",
        'set batch_dir = "$cwd"',
        "mkdir -p working output_mosaics logs",
        "if ( $status != 0 ) exit 1",
    ]

    for row in ready_list:
        (group_id, object_name, obs_date, telescope,
         band, img_start, img_end, flat_path) = row

        lab_date = date.fromisoformat(obs_date).strftime("%y_%m_%d")
        target = object_name.lower()

        filename = (
            f"{target}-{lab_date}-{telescope}-{band}"
            f"-flat-g{group_id}-mosaic.fits"
        )

        commands = make_mosaic_commands(
            group_id,
            telescope,
            lab_date,
            img_start,
            img_end,
            filename,
            flat_path,
        )

        output.append(commands)

    script_path = export_folder / "run_mosaics.csh"

    with script_path.open("w", encoding="utf-8") as file:
        file.write("\n".join(output) + "\n")

    return script_path

In [31]:
script_path = write_mosaic_script(ready_list, export_folder)

print("Script saved to:", script_path.resolve())
print("Number of groups:", len(ready_list))

Script saved to: /Users/mehtabsingh/Desktop/Desktop/Durham/3rd Year/Modules/Astro Lab/Astrolab_Code/lab batches/flat_01/run_mosaics.csh
Number of groups: 186
